# 🎧 AI Speech Intelligence System
## CS 5542 — Quiz Challenge 2 | Tina Nguyen

> **Full Pipeline:** Audio → Whisper STT → wav2vec2 STT (comparison) → NLP Analysis → MarianMT Translation → SpeechT5 TTS → Evaluation

**Runtime:** Select `Runtime → Change runtime type → T4 GPU` for best performance.

---

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install openai-whisper transformers accelerate sentencepiece datasets \
             keybert rouge-score sacrebleu nltk ffmpeg-python librosa soundfile \
             scikit-learn pandas numpy matplotlib gtts -q
!apt-get install -y ffmpeg -qq
print('✅ All dependencies installed!')

## 🎙️ Step 2: Provide or Generate Audio Input

You can either:
- **Option A:** Upload your own audio file
- **Option B:** Synthesize a demo lecture using gTTS

In [ ]:
# Option B: Generate synthetic lecture audio with gTTS
!pip install gtts -q

from gtts import gTTS
import pathlib

LECTURE_TEXT = """
Welcome everyone to today's session on artificial intelligence and natural language processing.
We are going to cover several important topics that are foundational to modern AI systems.

First, let's talk about transformer architectures. Transformers have revolutionized NLP since their
introduction in the paper Attention is All You Need by Vaswani et al. in 2017. The key innovation
is the self-attention mechanism, which allows the model to weigh the importance of different words
in a sequence when encoding a particular word.

We need to make sure everyone understands the concept of attention heads. Please review the assigned
reading before next class. Action item: complete the attention mechanism worksheet by Friday.

Moving on, let's discuss large language models, or LLMs. Models like GPT-4, Claude, and Llama 2
have demonstrated remarkable capabilities in text generation, reasoning, and even code synthesis.
However, we must also consider their limitations including hallucinations, bias, and computational cost.

For our next meeting, we should schedule a demo session where each team presents their RAG pipeline
results. Please make sure to prepare a five-minute presentation. The teaching assistants will follow
up with scheduling details.

Let's also briefly touch on evaluation metrics. For text generation tasks, we commonly use BLEU and
ROUGE scores. For information retrieval, precision and recall are key. Don't forget that automatic
metrics don't always correlate with human judgment.

To summarize today's key points: transformers use attention mechanisms, large language models are
powerful but have limitations, and evaluation requires multiple metrics.

Thank you all for your attention. Office hours are on Thursdays from 2 to 4 PM. Please submit your
assignment reports by Sunday midnight. See you next class.
"""

pathlib.Path('data').mkdir(exist_ok=True)
AUDIO_PATH = 'data/sample_lecture.mp3'

tts = gTTS(LECTURE_TEXT, lang='en', slow=False)
tts.save(AUDIO_PATH)
print(f'✅ Audio generated: {AUDIO_PATH}')

## 🤖 Step 3: Transcription — Baseline vs. Improved (Whisper)

In [ ]:
import whisper
import torch
import time
import json
import pathlib

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load Whisper base model
print('Loading Whisper base model...')
model = whisper.load_model('base', device=device)

# ── BASELINE TRANSCRIPTION ────────────────────────────────────────────────────
print('\n[1/2] Running BASELINE transcription...')
t0 = time.perf_counter()
result_baseline = model.transcribe(AUDIO_PATH, verbose=False)
baseline_text = result_baseline['text'].strip()
baseline_time = time.perf_counter() - t0

# ── IMPROVED TRANSCRIPTION ────────────────────────────────────────────────────
DOMAIN_PROMPT = (
    'This is a lecture or business meeting transcript. '
    'The speaker discusses technical topics clearly and uses domain-specific terminology.'
)

print('[2/2] Running IMPROVED transcription (domain prompt + beam_size=5)...')
t0 = time.perf_counter()
result_improved = model.transcribe(
    AUDIO_PATH,
    initial_prompt=DOMAIN_PROMPT,
    beam_size=5,
    temperature=0,
    condition_on_previous_text=True,
    verbose=False,
)
improved_text = result_improved['text'].strip()
improved_time = time.perf_counter() - t0

# ── Save outputs ──────────────────────────────────────────────────────────────
pathlib.Path('outputs/transcriptions').mkdir(parents=True, exist_ok=True)

for strategy, text, t in [('baseline', baseline_text, baseline_time), ('improved', improved_text, improved_time)]:
    with open(f'outputs/transcriptions/sample_lecture_{strategy}.json', 'w') as f:
        json.dump({'strategy': strategy, 'text': text, 'duration_s': round(t, 3),
                   'word_count': len(text.split())}, f, indent=2)

print(f'\n✅ Transcription complete!')
print(f'   Baseline  : {len(baseline_text.split())} words in {baseline_time:.1f}s')
print(f'   Improved  : {len(improved_text.split())} words in {improved_time:.1f}s')
print(f'\n── BASELINE (first 200 chars) ────────────────────────────────────')
print(baseline_text[:200])
print(f'\n── IMPROVED (first 200 chars) ────────────────────────────────────')
print(improved_text[:200])

## 🔁 Step 3a: wav2vec2-base-960h — Alternative STT Model

Compare `facebook/wav2vec2-base-960h` against Whisper. Note: output is ALL-CAPS with no punctuation.

In [ ]:
import torch, torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

W2V_MODEL = 'facebook/wav2vec2-base-960h'
print(f'Loading {W2V_MODEL}...')
w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_MODEL)
w2v_model = Wav2Vec2ForCTC.from_pretrained(W2V_MODEL)
if torch.cuda.is_available(): w2v_model = w2v_model.cuda()
w2v_model.eval()

waveform, sr = torchaudio.load(AUDIO_PATH)
if sr != 16000:
    waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
if waveform.shape[0] > 1: waveform = waveform.mean(0, keepdim=True)
audio_np = waveform.squeeze().numpy()

inputs = w2v_processor(audio_np, sampling_rate=16000, return_tensors='pt', padding=True).input_values
if torch.cuda.is_available(): inputs = inputs.cuda()
with torch.no_grad():
    logits = w2v_model(inputs).logits
wav2vec2_text = w2v_processor.decode(torch.argmax(logits, dim=-1)[0])

print(f'\n✅ wav2vec2 output ({len(wav2vec2_text.split())} words):')
print(wav2vec2_text[:300])
print('\n── WER Comparison ──')
print(f'  Whisper baseline WER : {baseline_wer:.4f}')
print(f'  Whisper improved WER : {improved_wer:.4f}')
wav2vec2_wer = wer(ref_transcript, wav2vec2_text.lower())
print(f'  wav2vec2 WER         : {wav2vec2_wer:.4f}')
print('  Note: wav2vec2 is ALL-CAPS — requires post-processing for downstream NLP.')

## 🧠 Step 4: NLP Analysis — Summary / Actions / Sentiment / Keywords

In [ ]:
import nltk
from collections import Counter
import re

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

ACTION_RE = re.compile(
    r'\b(we (need|should|must|will|have to)|'
    r'(todo|to-do|action item)[:\s]|'
    r'(follow[- ]up|follow up)|'
    r'(make sure|ensure|please|don\'t forget|remember to)|'
    r'(assign|schedule|send|review|check|confirm|update|prepare|complete|submit))\b',
    re.IGNORECASE,
)

print('✅ NLTK utilities ready')

In [ ]:
from transformers import pipeline
from keybert import KeyBERT

device_id = 0 if torch.cuda.is_available() else -1

print('Loading BART summarisation model...')
summarizer = pipeline('summarization', model='facebook/bart-large-cnn', device=device_id)

print('Loading DistilBERT sentiment model...')
sentiment_pipe = pipeline('sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english', device=device_id)

print('Loading KeyBERT...')
kw_model = KeyBERT()

print('✅ All NLP models loaded!')

In [ ]:
text = improved_text  # analyse the improved transcript
sentences = sent_tokenize(text)

# ── BASELINE ANALYSIS ─────────────────────────────────────────────────────────
baseline_summary = ' '.join(sentences[:3])
words = [w.lower() for w in word_tokenize(text) if w.isalpha() and w.lower() not in STOPWORDS]
freq = Counter(words).most_common(5)
baseline_keywords = [(w, round(c/len(words), 4)) for w, c in freq]
baseline_sentiment = sentiment_pipe(text[:512])[0]

# ── IMPROVED ANALYSIS ─────────────────────────────────────────────────────────
# 1. Abstractive summary
summary_out = summarizer(text[:3000], max_length=180, min_length=40, do_sample=False)
improved_summary = summary_out[0]['summary_text']

# 2. Action items
action_items = [s.strip() for s in sentences if ACTION_RE.search(s)][:10]

# 3. Per-sentence sentiment
sent_labels = [sentiment_pipe(s[:512])[0] for s in sentences[:20]]
labels = [r['label'] for r in sent_labels]
scores = [r['score'] for r in sent_labels]
dominant = Counter(labels).most_common(1)[0][0]
improved_sentiment = {
    'label': dominant,
    'score': round(sum(scores)/len(scores), 4),
    'positive_ratio': round(labels.count('POSITIVE')/len(labels), 4),
    'sentence_count': len(labels)
}

# 4. KeyBERT keyphrases
improved_keywords = kw_model.extract_keywords(
    text, keyphrase_ngram_range=(1, 2), stop_words='english',
    use_mmr=True, diversity=0.5, top_n=10
)

print('✅ Analysis complete!')
print(f'   Sentences: {len(sentences)}')
print(f'   Action items: {len(action_items)}')
print(f'   Keywords: {len(improved_keywords)}')

## 📊 Step 5a: Display NLP Results

In [ ]:
print('='*65)
print('  BASELINE SUMMARY (extractive — first 3 sentences)')
print('='*65)
print(baseline_summary)

print()
print('='*65)
print('  IMPROVED SUMMARY (abstractive — BART-large-CNN)')
print('='*65)
print(improved_summary)

print()
print('='*65)
print('  ACTION ITEMS DETECTED (Improved only)')
print('='*65)
if action_items:
    for i, item in enumerate(action_items, 1):
        print(f'  {i}. {item}')
else:
    print('  (none detected)')

print()
print('='*65)
print('  SENTIMENT ANALYSIS')
print('='*65)
print(f'  Baseline : {baseline_sentiment}')
print(f'  Improved : {improved_sentiment}')

print()
print('='*65)
print('  KEYWORD EXTRACTION')
print('='*65)
print('  Baseline (TF-IDF word freq):')
for kw, score in baseline_keywords:
    print(f'    {kw:<25} {score:.4f}')
print()
print('  Improved (KeyBERT + MMR):')
for kw, score in improved_keywords:
    print(f'    {kw:<25} {score:.4f}')

## 📏 Step 6a: Quantitative Evaluation (Transcription Quality, Summary Quality, Sentiment Usefulness, Latency)

In [ ]:
import numpy as np
from rouge_score import rouge_scorer as rs
import pandas as pd

# Load reference data
try:
    ref_transcript = open('data/sample_reference_transcript.txt').read()
    ref_summary = open('data/sample_reference_summary.txt').read()
except FileNotFoundError:
    ref_transcript = LECTURE_TEXT
    ref_summary = 'Transformers use self-attention mechanisms and revolutionized NLP. Large language models are powerful but face hallucination and bias. Key metrics include ROUGE and BLEU.'

# ── WER ───────────────────────────────────────────────────────────────────────
def wer(ref, hyp):
    ref_w = ref.lower().split()
    hyp_w = hyp.lower().split()
    n, m = len(ref_w), len(hyp_w)
    dp = np.zeros((n+1, m+1), dtype=np.int32)
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref_w[i-1] == hyp_w[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return round(dp[n][m] / n, 4) if n > 0 else 0.0

# ── ROUGE ─────────────────────────────────────────────────────────────────────
scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def rouge(ref, hyp):
    s = scorer.score(ref, hyp)
    return {k: round(v.fmeasure, 4) for k, v in s.items()}

baseline_wer = wer(ref_transcript, baseline_text)
improved_wer = wer(ref_transcript, improved_text)
baseline_rouge = rouge(ref_summary, baseline_summary)
improved_rouge = rouge(ref_summary, improved_summary)

rows = [
    ['WER ↓ (lower is better)', f'{baseline_wer:.4f}', f'{improved_wer:.4f}'],
    ['ROUGE-1 ↑', baseline_rouge['rouge1'], improved_rouge['rouge1']],
    ['ROUGE-2 ↑', baseline_rouge['rouge2'], improved_rouge['rouge2']],
    ['ROUGE-L ↑', baseline_rouge['rougeL'], improved_rouge['rougeL']],
    ['Keywords extracted', len(baseline_keywords), len(improved_keywords)],
    ['Action items detected', 0, len(action_items)],
    ['Sentiment (overall)', baseline_sentiment['label'], improved_sentiment['label']],
]

df = pd.DataFrame(rows, columns=['Metric', 'Baseline', 'Improved'])
print()
print('='*60)
print('  EVALUATION RESULTS: BASELINE vs. IMPROVED')
print('='*60)
print(df.to_string(index=False))
print('='*60)
print('\n── 3. Sentiment Usefulness ──')
print(f'  Baseline Sentiment: {baseline_sentiment}')
print(f'  Improved Sentiment: {improved_sentiment}')
print('  (Improved per-sentence sentiment is significantly more useful for extracting precise action items)')

print('\n── 4. Latency ──')
print(f'  Baseline STT: {baseline_duration_s:.2f} seconds')
print(f'  Improved STT: {improved_duration_s:.2f} seconds')
print('  (Note: Baseline is faster because beam_search=5 adds latency to Improved, but vastly increases quality)')


## 📈 Step 7: Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Speech Intelligence System\nBaseline vs. Improved Comparison', fontsize=14, fontweight='bold')

# ── Plot 1: ROUGE scores ──────────────────────────────────────────────────────
ax = axes[0]
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
b_vals = [baseline_rouge['rouge1'], baseline_rouge['rouge2'], baseline_rouge['rougeL']]
i_vals = [improved_rouge['rouge1'], improved_rouge['rouge2'], improved_rouge['rougeL']]
x = range(len(metrics))
ax.bar([xi - 0.2 for xi in x], b_vals, 0.35, label='Baseline', color='#6C757D', alpha=0.85)
ax.bar([xi + 0.2 for xi in x], i_vals, 0.35, label='Improved', color='#0066CC', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(metrics)
ax.set_ylim(0, 0.7)
ax.set_title('ROUGE Scores (Summarisation)')
ax.set_ylabel('F1 Score')
ax.legend()
ax.grid(axis='y', alpha=0.4)

# ── Plot 2: WER ───────────────────────────────────────────────────────────────
ax = axes[1]
ax.bar(['Baseline', 'Improved'], [baseline_wer, improved_wer],
       color=['#6C757D', '#0066CC'], alpha=0.85)
ax.set_title('Word Error Rate (WER) ↓')
ax.set_ylabel('WER (lower is better)')
ax.set_ylim(0, max(baseline_wer, improved_wer) * 1.4)
for i, v in enumerate([baseline_wer, improved_wer]):
    ax.text(i, v + 0.002, f'{v:.4f}', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.4)

# ── Plot 3: Feature comparison ────────────────────────────────────────────────
ax = axes[2]
features = ['Keywords', 'Action Items']
b_counts = [len(baseline_keywords), 0]
i_counts = [len(improved_keywords), len(action_items)]
x = range(len(features))
ax.bar([xi - 0.2 for xi in x], b_counts, 0.35, label='Baseline', color='#6C757D', alpha=0.85)
ax.bar([xi + 0.2 for xi in x], i_counts, 0.35, label='Improved', color='#0066CC', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(features)
ax.set_title('Feature Extraction Count')
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
pathlib.Path('outputs/evaluation').mkdir(parents=True, exist_ok=True)
plt.savefig('outputs/evaluation/comparison_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Charts saved to outputs/evaluation/comparison_charts.png')

## 🌐 Step 5: Multilingual Translation — Helsinki-NLP MarianMT

Translate the improved summary to Spanish, French, and Vietnamese.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import pathlib, json

LANG_MODELS = {
    'es': ('Spanish',    'Helsinki-NLP/opus-mt-en-es'),
    'fr': ('French',     'Helsinki-NLP/opus-mt-en-fr'),
    'vi': ('Vietnamese', 'Helsinki-NLP/opus-mt-en-vi'),
}

text_to_translate = improved_summary
translations = {}
pathlib.Path('outputs/translations').mkdir(parents=True, exist_ok=True)

for lang, (lang_name, model_id) in LANG_MODELS.items():
    print(f'Translating to {lang_name} ({model_id})...')
    tok = MarianTokenizer.from_pretrained(model_id)
    mdl = MarianMTModel.from_pretrained(model_id)
    inputs = tok([text_to_translate], return_tensors='pt', padding=True, truncation=True, max_length=512)
    ids = mdl.generate(**inputs, num_beams=4, early_stopping=True)
    translated = tok.batch_decode(ids, skip_special_tokens=True)[0]
    translations[lang] = translated
    result = {'target_lang': lang, 'language_name': lang_name, 'model_id': model_id,
              'original_text': text_to_translate, 'translated_text': translated,
              'word_count_original': len(text_to_translate.split()),
              'word_count_translated': len(translated.split())}
    with open(f'outputs/translations/sample_lecture_{lang}.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f'  [{lang_name}]: {translated[:150]}...')

print('\n✅ Translation complete!')

## 🔊 Step 6: Text-to-Speech — microsoft/speecht5_tts

Convert the improved summary back to speech audio. Closes the full loop: **Audio IN → STT → NLP → Translation → TTS → Audio OUT**.

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import soundfile as sf, torch, pathlib

print('Loading SpeechT5 TTS model...')
tts_processor = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
tts_model     = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts')
vocoder       = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan')
if torch.cuda.is_available():
    tts_model = tts_model.cuda(); vocoder = vocoder.cuda()

print('Loading speaker embeddings...')
embeddings_ds = load_dataset('Matthijs/cmu-arctic-xvectors', split='validation')
speaker_embeddings = torch.tensor(embeddings_ds[7306]['xvector']).unsqueeze(0)
if torch.cuda.is_available(): speaker_embeddings = speaker_embeddings.cuda()

text_in = improved_summary[:600]
print(f'Synthesizing {len(text_in)} chars...')
inputs = tts_processor(text=text_in, return_tensors='pt')
if torch.cuda.is_available(): inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    speech = tts_model.generate_speech(inputs['input_ids'], speaker_embeddings, vocoder=vocoder)

pathlib.Path('outputs/tts').mkdir(parents=True, exist_ok=True)
out_path = 'outputs/tts/sample_lecture_summary.wav'
sf.write(out_path, speech.cpu().numpy(), samplerate=16000)
print(f'✅ Audio saved: {out_path}  ({len(speech)/16000:.1f}s)')

# Play in Colab
from IPython.display import Audio, display
display(Audio(out_path))

## 💾 Step 8: Save All Results to JSON

In [ ]:
final_report = {
    'audio_file': AUDIO_PATH,
    'whisper_model': 'base',
    'transcription': {
        'baseline_word_count': len(baseline_text.split()),
        'improved_word_count': len(improved_text.split()),
        'baseline_wer': baseline_wer,
        'improved_wer': improved_wer,
    },
    'summarisation': {
        'baseline_rouge': baseline_rouge,
        'improved_rouge': improved_rouge,
        'baseline_summary': baseline_summary,
        'improved_summary': improved_summary,
    },
    'sentiment': {
        'baseline': {'label': baseline_sentiment['label'], 'score': round(baseline_sentiment['score'], 4)},
        'improved': improved_sentiment,
    },
    'keywords': {
        'baseline': [{'phrase': kw, 'score': score} for kw, score in baseline_keywords],
        'improved': [{'phrase': kw, 'score': round(score, 4)} for kw, score in improved_keywords],
    },
    'action_items': {
        'baseline_count': 0,
        'improved_count': len(action_items),
        'items': action_items,
    },
}

with open('outputs/evaluation/full_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print('✅ Full report saved to outputs/evaluation/full_report.json')
print()
print('📁 Output structure:')
import os
for root, dirs, files in os.walk('outputs'):
    level = root.replace('outputs', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')